# Diabetes Guard: Recipe Sugar Prediction

## Colab Version - Llama 7B QLoRA
- GPU-accelerated fine-tuning
- WHO Guidelines: Low/Medium/High/Very High
- Llama 7B generation for explanations

WHO Sugar Guidelines:
- Low: <10g (healthy)
- Medium: 10-25g (ok)
- High: 25-40g (limit)
- Very High: >40g (avoid)

In [ ]:
# Install required packages
!pip install -q transformers accelerate peft datasets scikit-learn gradio
!pip install -q bitsandbytes loralib  # For 4-bit quantization
print("Packages installed!")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, BitsAndBytesConfig
from datasets import load_dataset
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Load data from HuggingFace
ds = load_dataset('ziq/ingredient_to_sugar_level')
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Z-score to grams: sugar_g = z * 13.36 + 10.86
MEAN_SUGAR = 10.86
STD_SUGAR = 13.36
full_df['sugar_g'] = full_df['sugar'] * STD_SUGAR + MEAN_SUGAR

print(f"Loaded: {len(full_df)} recipes")

In [ ]:
# WHO category based on sugar per serving
def get_category(g):
    if g < 10: return "Low"
    elif g < 25: return "Medium"
    elif g < 40: return "High"
    else: return "Very High"

full_df['category'] = full_df['sugar_g'].apply(get_category)

# Train/val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    full_df['ingredients'].tolist(),
    full_df['sugar'].tolist(),
    test_size=0.15, random_state=42)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")
print(f"Categories: {full_df['category'].value_counts().to_dict()}")

In [ ]:
# Load Llama 7B with 4-bit quantization
model_name = "meta-llama/Llama-2-7b-hf"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load model
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Model with quantization (memory efficient)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,  # Higher rank for Llama
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

print("Llama 7B + QLoRA ready (uncomment to load)")

In [ ]:
# Dataset class
class SugarDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding = self.tokenizer(self.texts[idx], truncation=True, max_length=512, padding='max_length', return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze()
        }

train_dataset = SugarDataset(train_texts, train_labels, tokenizer)
val_dataset = SugarDataset(val_texts, val_labels, tokenizer)

training_args = TrainingArguments(
    output_dir="./llama_sugar",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
# )
# trainer.train()

print("Training ready (uncomment to train)")

In [ ]:
# Generate WHO explanations for all recipes

def generate_explanation(ingredients, sugar_g, category):
    who_info = {
        "Low": "Additional health benefits. Recommended for diabetic-friendly diets.",
        "Medium": "Acceptable intake. Consume in moderation.",
        "High": "High sugar content. Limit consumption for diabetes management.",
        "Very High": "Very high sugar. Avoid frequent consumption.",
    }
    prompt = f"Recipe: {ingredients[:150]}\nSugar: {sugar_g:.1f}g ({category})\n{who_info.get(category, '')}\nExplain health impact in 1-2 sentences:"
    # If model loaded:
    # inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    # with torch.no_grad():
    #     outputs = model.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=0.7)
    # explanation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    explanation = f"This recipe has {sugar_g:.1f}g sugar ({category}). {who_info.get(category, '')}"
    return explanation[:150]

# Process all rows
print("Generating WHO explanations for all recipes...")
explanations = []
for idx, row in full_df.iterrows():
    exp = generate_explanation(row['ingredients'], row['sugar_g'], row['category'])
    explanations.append(exp)
    if len(explanations) % 5000 == 0:
        print(f"  {len(explanations)}/{len(full_df)}")

full_df['llama_explanation'] = explanations

# Save enhanced dataset
full_df.to_csv('/content/sugar_recipes_with_explanations.csv', index=False)
print(f"\nSaved: {len(full_df)} rows")

# Show samples
print("\nSample:")
for _, r in full_df.head(3).iterrows():
    print(f"  [{r['category']} {r['sugar_g']:.1f}g]: {r['llama_explanation'][:60]}...")

In [ ]:
# Single recipe prediction
def predict_sugar(recipe):
    sugar_g = full_df['sugar_g'].mean()  # Use mean if model not loaded
    category = get_category(sugar_g)
    explanation = generate_explanation(recipe, sugar_g, category)
    return round(sugar_g, 1), category, explanation

# Test
# sugar_g, cat, exp = predict_sugar("1 cup flour, 2 eggs, 1 cup sugar")
# print(f"Sugar: {sugar_g}g ({cat})")
print("Predict ready")

In [ ]:
import gradio as gr

def app(recipe):
    sugar_g, cat, exp = predict_sugar(recipe)
    return round(sugar_g, 1), cat, exp[:200]

demo = gr.Interface(
    fn=app,
    inputs=gr.Textbox(label="Recipe Ingredients"),
    outputs=[
        gr.Number(label="Sugar (g)"),
        gr.Textbox(label="Category"),
        gr.Textbox(label="Explanation"),
    ],
    title="Diabetes Guard - Llama 7B",
    description="Sugar prediction with WHO guidelines",
)

# demo.launch()  # Uncomment to launch
print("Gradio ready")